# Finding ancestors of conditionality in SPDX

### 1. Setting up

Conditions: Manually extracted list of all conditions in SPDX 3.0

Schema: My manually extracted schema from the SPDX specifications. I chose the manually extracted schema over the official schema at this stage in order to be consistent with my other work, which had findings based off of my own schema.

In [7]:
import json

CLASSES_FILE = 'classes_core.txt'
SCHEMA_FILE = 'schema.json'
OUTPUT_FILE = "classes_ancestors.csv"


### 2. Loading classes and schema

In [3]:
with open(CLASSES_FILE, 'r') as f:
    classes = {line.strip() for line in f}

print(f"{len(classes)}  classes loaded: {classes}\n")

with open(SCHEMA_FILE, 'r') as f:
    schema = json.load(f)

20  classes loaded: {'Hash', 'Annotation', 'IndividualElement', 'Person', 'PositiveIntegerRange', 'Relationship', 'ElementCollection', 'Bom', 'NamespaceMap', 'PackageVerificationCode', 'ExternalMap', 'LifecycleScopedRelationship', 'Bundle', 'Organization', 'CreationInfo', 'Element', 'Agent', 'ExternalIdentifier', 'DictionaryEntry', 'Artifact'}



### 3. Finding all conditional classes

In [4]:
ancestor_graph = {}
all_nodes = set(schema["$defs"].keys())

for parent_name, definition in schema["$defs"].items():
    def find_refs_in_definition(obj):
        refs = set()
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key == "$ref" and isinstance(value, str):
                    child_name = value.split('/')[-1]
                    if child_name in all_nodes:
                        refs.add(child_name)
                else:
                    refs.update(find_refs_in_definition(value))
        elif isinstance(obj, list):
            for item in obj:
                refs.update(find_refs_in_definition(item))
        return refs

    children = find_refs_in_definition(definition)
    for child in children:
        if child not in ancestor_graph:
            ancestor_graph[child] = set()
        ancestor_graph[child].add(parent_name)

# --- MODIFIED PART STARTS HERE ---

final_results = {}
for target_class in classes:
    if target_class not in all_nodes:
        final_results[target_class] = {'ancestors': set(), 'count': 0, 'error': 'Class not found in schema definitions.'}
        continue

    # Set to store all unique ancestors found during traversal.
    all_ancestors = set()
    # Set to keep track of visited nodes to prevent infinite loops in case of cyclic dependencies.
    visited_nodes = set()

    def find_all_ancestors_recursive(current_node):
        """
        Recursively traverses the graph upwards from a node,
        collecting all unique ancestors.
        """
        # If we've already processed this node, stop to avoid cycles.
        if current_node in visited_nodes:
            return

        visited_nodes.add(current_node)

        # Check if the current node has any parents in our graph.
        if current_node in ancestor_graph:
            direct_parents = ancestor_graph[current_node]
            
            # Add the direct parents to our collection of all ancestors.
            all_ancestors.update(direct_parents)
            
            # Recursively find the ancestors of each parent.
            for parent in direct_parents:
                find_all_ancestors_recursive(parent)

    # Start the recursive search from the target class.
    find_all_ancestors_recursive(target_class)

    # Store the final set of ancestors and their count.
    final_results[target_class] = {
        'ancestors': all_ancestors,
        'count': len(all_ancestors)
    }

### 4. Printing results

In [5]:
print("--- Ancestor Count Analysis ---")
for class_name, result in final_results.items():
    print(f"\nClass: '{class_name}'")
    if 'error' in result:
        print(f"  Error: {result['error']}")
        continue

    # Get the count and the set of ancestors from the result dictionary
    count = result['count']
    ancestors = result['ancestors']
    
    print(f"  Number of Unique Ancestors: {count}")
    
    # Optionally, print the list of ancestors.
    # Sorting them makes the output consistent and easier to read.
    if ancestors:
        ancestor_list_str = ", ".join(sorted(list(ancestors)))
        print(f"  Ancestors: {ancestor_list_str}")
    else:
        print("  Ancestors: None")

--- Ancestor Count Analysis ---

Class: 'Hash'
  Number of Unique Ancestors: 1
  Ancestors: build_Build

Class: 'Annotation'
  Number of Unique Ancestors: 0
  Ancestors: None

Class: 'IndividualElement'
  Number of Unique Ancestors: 2
  Ancestors: NoAssertionElement, NoneElement

Class: 'Person'
  Number of Unique Ancestors: 0
  Ancestors: None

Class: 'PositiveIntegerRange'
  Number of Unique Ancestors: 1
  Ancestors: software_Snippet

Class: 'Relationship'
  Number of Unique Ancestors: 13
  Ancestors: LifecycleScopedRelationship, security_CvssV2VulnAssessmentRelationship, security_CvssV3VulnAssessmentRelationship, security_CvssV4VulnAssessmentRelationship, security_EpssVulnAssessmentRelationship, security_ExploitCatalogVulnAssessmentRelationship, security_SsvcVulnAssessmentRelationship, security_VexAffectedVulnAssessmentRelationship, security_VexFixedVulnAssessmentRelationship, security_VexNotAffectedVulnAssessmentRelationship, security_VexUnderInvestigationVulnAssessmentRelationship

### 5. Saving results

In [8]:
import csv

# Define the output file name (replace with your actual file name)
OUTPUT_FILE = 'ancestor_analysis.csv' 

with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as csvfile:
    # Create a writer object
    writer = csv.writer(csvfile)

    # Write the new header row to reflect the new data
    writer.writerow(['class_name', 'ancestor_count', 'ancestors'])

    # Loop through your dictionary and write the new data format
    for class_name, result in final_results.items():
        if 'error' in result:
            # Write a row for an error case, placing the error message in the 'ancestors' column
            writer.writerow([class_name, 'ERROR', result['error']])
        else:
            # Get the count and the set of ancestors
            count = result['count']
            ancestors_set = result['ancestors']

            # Format the set of ancestors into a single, sorted, comma-separated string for the CSV cell
            ancestors_string = ", ".join(sorted(list(ancestors_set)))
            
            # Write the row for a success case
            writer.writerow([class_name, count, ancestors_string])

print(f"Ancestor analysis results have been saved to {OUTPUT_FILE}")

Ancestor analysis results have been saved to ancestor_analysis.csv
